# 02 · Preprocess + NLP enrichment

Pipeline: clean → language filter (en) → dedup (exact + near) → sentiment (VADER+TextBlob) → entity extraction → region tag.

Output: `data/processed/twitter_enriched.csv`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd

from config.settings import DATA_RAW, DATA_PROCESSED
from preprocessing.text_cleaner import process_dataframe
from preprocessing.language_filter import filter_language
from preprocessing.deduplicator import deduplicate
from nlp.sentiment import SentimentAnalyzer
from nlp.entity_extractor import enrich

raw = pd.read_csv(DATA_RAW / 'twitter_raw.csv')
print(f'Raw: {len(raw)}')
raw.head(2)

Raw: 5610


,tweet_id,text,created_at,user_name,user_screen_name,user_followers,favorite_count,retweet_count,reply_count,language,query,query_group,scraped_at
0,2056177448127967240,🔥 TRENDING NOW! 🔥\nZulay Kitchen Rechargeable ...,Mon May 18 00:58:00 +0000 2026,NaN,NaN,800,0,0,0,en,matcha protein drink,flavor,2026-05-18T19:59:10.383352
1,2056077592982147337,@V4MPlREM0NEY ja 100% \nich hab auch nur rossm...,Sun May 17 18:21:12 +0000 2026,NaN,NaN,356,1,0,1,de,matcha protein drink,flavor,2026-05-18T19:59:10.383387


In [2]:
# Clean text
df = process_dataframe(raw, text_col='text')
print(f'After cleaning: {len(df)}')

After cleaning: 5609


In [3]:
# Language filter (keep English)
df = filter_language(df, text_col='clean_text', keep='en')
print(f'After lang filter: {len(df)}')

After lang filter: 5416


In [4]:
# Dedup
df = deduplicate(df, text_col='clean_text')
print(f'After dedup: {len(df)}')

After dedup: 5021


In [5]:
# Sentiment
analyzer = SentimentAnalyzer()
df = analyzer.analyze_dataframe(df, text_col='clean_text')
df['sentiment_label'].value_counts()

sentiment_label
positive    3004
negative    1450
neutral      567
Name: count, dtype: int64

In [6]:
# Entity extraction (flavors / pains / occasions / formats / brands / region)
df = enrich(df, text_col='clean_text')
df[['n_flavors', 'n_pains', 'n_occasions', 'n_formats', 'n_brands']].sum()

n_flavors      2754
n_pains         521
n_occasions    2049
n_formats      2762
n_brands        495
dtype: int64

In [7]:
# Save
out_path = DATA_PROCESSED / 'twitter_enriched.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df)} -> {out_path}')
df.head(2)

Saved 5021 -> /home/azril/Personal/Projects/DSAI/NUSFTC/nlp social listening 2/data/processed/twitter_enriched.csv


,tweet_id,text,created_at,user_name,user_screen_name,user_followers,favorite_count,retweet_count,reply_count,language,...,pains,occasions,formats,brands,region_tag,n_flavors,n_pains,n_occasions,n_formats,n_brands
0,2056177448127967240,🔥 TRENDING NOW! 🔥\nZulay Kitchen Rechargeable ...,Mon May 18 00:58:00 +0000 2026,NaN,NaN,800,0,0,0,en,...,[],[],[powder],[],Global,2,0,0,1,0
1,2056054474981232872,Zulay Kitchen Milk Frother Handheld Electric W...,Sun May 17 16:49:21 +0000 2026,NaN,NaN,7523,0,0,0,en,...,[],[],[powder],[],Global,2,0,0,1,0
